In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ==========================
# LOAD DATA
# ==========================

file="../3.Near Miss Database - 2012.xlsx"
sheet="Near Miss"

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

original_shape=df.shape

print("Original Shape:",original_shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# REMOVE FULLY EMPTY COLUMNS
# ==========================

keep=[]

for col in df.columns:

    if col.lower()=="si_no":
        keep.append(col)

    elif df[col].notna().sum()>0:
        keep.append(col)

df=df[keep]

# ==========================
# CREATE SI NO IF EMPTY
# ==========================

si_col=None

for col in df.columns:

    clean=col.lower().replace("_","").replace(" ","")

    if clean in ["sino","slno","serialno"]:

        si_col=col
        break

if si_col:

    if (
        df[si_col]
        .replace("",np.nan)
        .isna()
        .all()
    ):

        df[si_col]=range(
            1,
            len(df)+1
        )

# ==========================
# REMOVE EMPTY REF NO
# ==========================

ref_col=None

for col in df.columns:

    if "ref" in col.lower():

        ref_col=col
        break

if ref_col:

    before=len(df)

    df=df[
        df[ref_col]
        .fillna("")
        .str.strip()
        !=""
    ]

    print(
        "Rows Removed (Missing Ref_No):",
        before-len(df)
    )

# ==========================
# REMOVE EMPTY DESCRIPTION
# ==========================

desc=None

for col in df.columns:

    if "description" in col.lower():

        desc=col
        break

if desc:

    before=len(df)

    df=df[
        df[desc]
        .fillna("")
        .str.strip()
        !=""
    ]

    print(
        "Rows Removed (Missing Description):",
        before-len(df)
    )

# ==========================
# MERGE POSSIBLE CONSEQUENCE
# ==========================

pc=[]

for col in df.columns:

    if (
        "possible" in col.lower()
        or
        "unnamed" in col.lower()
    ):
        pc.append(col)

if pc:

    merged=(
        df[pc]
        .fillna("")
        .apply(
            lambda x:
            " | ".join(
                [
                    i.strip()
                    for i in x
                    if str(i).strip()
                    and str(i).lower()!="n/a"
                ]
            ),
            axis=1
        )
    )

    first=pc[0]

    df[first]=merged

    remove=[
        c
        for c in pc[1:]
        if "unnamed" in c.lower()
    ]

    df=df.drop(
        columns=remove,
        errors="ignore"
    )

# ==========================
# DATE CLEANING
# ==========================

date_cols=[]

keywords=[
    "date",
    "target"
]

for col in df.columns:

    if any(
        k in col.lower()
        for k in keywords
    ):
        date_cols.append(col)

def clean_date(v):

    if pd.isna(v):
        return np.nan

    v=str(v).strip()

    if v.lower() in [
        "na",
        "n/a",
        "not mentioned",
        "nan"
    ]:
        return np.nan

    if v=="":
        return np.nan

    # already correct → keep
    if re.match(
        r"^\d{2}-[A-Za-z]{3}-\d{2}$",
        v
    ):
        return v

    # keep values like 30-Sep
    if re.match(
        r"^\d{1,2}-[A-Za-z]{3}$",
        v
    ):
        return v

    try:

        d=pd.to_datetime(
            v,
            errors="coerce"
        )

        if pd.notna(d):

            return d.strftime(
                "%d-%b-%y"
            )

    except:
        pass

    return np.nan

for col in date_cols:

    df[col]=(
        df[col]
        .apply(clean_date)
    )

# ==========================
# REMOVE DUPLICATES
# ==========================

dup=df.duplicated().sum()

df=df.drop_duplicates()

print("Duplicates:",dup)

# ==========================
# RESET SI NO AFTER CLEANING
# ==========================

si_col=None

for col in df.columns:

    clean=(
        col
        .lower()
        .replace("_","")
        .replace(" ","")
        .replace(":","")
    )

    if clean in [
        "sino",
        "slno",
        "serialno"
    ]:

        si_col=col
        break

if si_col:

    df=df.reset_index(
        drop=True
    ).copy()

    df.loc[:,si_col]=np.arange(
        1,
        len(df)+1
    )

print("\nSI_No regenerated")
# ==========================
# MISSING SUMMARY
# ==========================

missing=(
    df
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Summary:")
print(missing)

# ==========================
# FINAL OUTPUT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================
df=df.reset_index(
    drop=True
)
out="cleaned_3_Near_Miss.xlsx"

df.to_excel(
    out,
    index=False
)

print("\nSaved:",out)

Original Shape: (500, 31)
Rows Removed (Missing Ref_No): 1
Rows Removed (Missing Description): 4


C:\Users\vinyt\AppData\Local\Temp\ipykernel_32556\1677458987.py:240: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  d=pd.to_datetime(


Duplicates: 0

SI_No regenerated

Missing Summary:
Type_of_Ship                                1
Incident_Category_(Potential)             108
Location_of_the_Unsafe_act_/_condition     12
Probability_of_Reoccurence                106
Cause_Analysis                              9
Counter_measure_to_prevent_recurrence       3
Date_Corrective_action_completed          221
Area_of_Concern                           132
Date_Investigation_Commenced              118
Target_Date                                61
Date_of_Investigation_Completion          139
Extension_(if_any)_with_remarks           477
Time_Period_(in_days)                     454
Master                                    106
Chief_Engineer                            137
Superintendent                             70
dtype: int64

Final Shape:
(495, 27)

Columns:
['Sl_No:', 'Ref.No.', 'Name_of_Ship', 'Type_of_Ship', 'Date_of_the_Near_Miss', 'Date_Reported', 'Delay_(in_days)', 'Period', 'Description', 'Incident_Category_(Potent